> **LEGACY NOTEBOOK (kept as documentation of the scraping step) – superseded by `02.4_resort_data_repair.ipynb`.**
>
> The audit of 28 Nov 2026 found problems in the output of this workflow (hand-added rows with median values, `lifts_count` artefact, wrong `slope_km`, formula instead of hotel counts, coordinates lost in a name-based merge). To avoid overwriting the repaired tables, this notebook now writes to `*_legacy_quota75.csv` files. The valid tables are `data/processed/resorts_clean.csv` and `resorts_features.csv`, produced by notebook 02.4.


### 1. Environment Setup & Discovery Crawler
Initializes folder structures (`data/raw/html_resorts/` and `data/processed/`) and executes a multi-page discovery crawler across national indices (CH, IT, AT, FR) to gather ~75 major ski resorts per country (~300 total) adhering to polite request intervals (2.5s) and academic User-Agent identification.

In [18]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Directory Setup
RAW_HTML_DIR = "data/raw/html_resorts"
PROCESSED_DIR = "data/processed"
os.makedirs(RAW_HTML_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 2. Ethical Scraping Settings
HEADERS = {
    "User-Agent": "ZHAW-DataAnalytics-Research-StudentProject/1.0 (contact: student@zhaw.ch)",
    "Accept-Language": "en-US,en;q=0.9"
}
REQUEST_DELAY_SECONDS = 2.0
BASE_URL = "https://www.skiresort.info"

COUNTRIES = [
    {"code": "CH", "path": "/ski-resorts/switzerland/"},
    {"code": "IT", "path": "/ski-resorts/italy/"},
    {"code": "AT", "path": "/ski-resorts/austria/"},
    {"code": "FR", "path": "/ski-resorts/france/"}
]

resorts_to_scrape = []
print("--> Starting automated URL discovery across Switzerland, Italy, Austria, and France...\n")

for c in COUNTRIES:
    c_code = c["code"]
    country_collected = 0
    seen_urls = set()
    print(f"Collecting resorts for {c_code}...")

    # Traversa le prime 4 pagine dell'indice nazionale
    for page_num in range(1, 5):
        if country_collected >= 75:
            break

        if page_num == 1:
            page_url = f"{BASE_URL}{c['path']}"
        else:
            page_url = f"{BASE_URL}{c['path']}page/{page_num}/"

        try:
            res = requests.get(page_url, headers=HEADERS, timeout=15)
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")
                
                # Cerca tutti i link che puntano a una scheda di comprensorio (/ski-resort/...)
                for a in soup.find_all("a", href=True):
                    href = a["href"].strip()
                    
                    # Normalizza a URL assoluto se relativo
                    if href.startswith("/"):
                        full_resort_url = f"{BASE_URL}{href}"
                    else:
                        full_resort_url = href

                    # Verifica che sia un link valido a un comprensorio
                    if "/ski-resort/" in full_resort_url and not any(sub in full_resort_url for sub in [
                        "page/", "search", "sorted", "resorts", "regions", 
                        "test-result", "trail-map", "photos", "weather", "webcams", "hotel"
                    ]):
                        # Rimuove parametri query o slash finali per deduplica
                        clean_url = full_resort_url.split("?")[0].rstrip("/") + "/"
                        
                        # Verifica che il link abbia la struttura canonica: .../ski-resort/{nome}/
                        parts = [p for p in clean_url.split("/") if p]
                        if len(parts) >= 4 and parts[-2] == "ski-resort":
                            if clean_url not in seen_urls:
                                name = a.get_text().strip()
                                # Se il link non ha testo diretto, ricava il nome dallo slug URL
                                if len(name) < 3 or "\n" in name:
                                    name = parts[-1].replace("-", " ").title()

                                seen_urls.add(clean_url)
                                resorts_to_scrape.append({
                                    "name": name,
                                    "country": c_code,
                                    "url": clean_url
                                })
                                country_collected += 1
                                if country_collected >= 75:
                                    break
            time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as e:
            print(f"  [WARN] Issue fetching {page_url}: {e}")
            break

    print(f"  [OK] Found {country_collected} valid resorts for {c_code}.")

print("\n" + "=" * 65)
print(f"CRAWLER SUMMARY: Collected {len(resorts_to_scrape)} target resort URLs.")
print("=" * 65)

--> Starting automated URL discovery across Switzerland, Italy, Austria, and France...

  [OK] Found 75 valid resorts for CH.
  [OK] Found 75 valid resorts for IT.
  [OK] Found 75 valid resorts for AT.
  [OK] Found 75 valid resorts for FR.

CRAWLER SUMMARY: Collected 300 target resort URLs.


### 2. Main HTML Caching & Variable Extraction Loop
Iterates through discovered destination URLs, caching raw HTML to `data/raw/html_resorts/` to prevent redundant network calls. Uses BeautifulSoup and NFKD unicode normalization to extract elevations (base and top in meters), slope kilometers, lifts count, adult high-season prices, star ratings, and artificial snow cannons.

In [19]:
scraped_records = []
print(f"--> Re-parsing {len(resorts_to_scrape)} local HTML files with calibrated patterns...\n")

for idx, item in enumerate(resorts_to_scrape):
    # Sanitize URL slug to locate cached files on disk
    url_slug = [p for p in item["url"].split("/") if p][-1]
    slug = re.sub(r"[^a-z0-9]+", "_", url_slug.lower())[:60]
    raw_html_path = os.path.join(RAW_HTML_DIR, f"{slug}.html")

    # Fallback to full name slug if cached under the previous naming scheme
    if not os.path.exists(raw_html_path):
        old_slug = re.sub(r"[^a-z0-9]+", "_", item["name"].lower().strip("_"))
        raw_html_path = os.path.join(RAW_HTML_DIR, f"{old_slug}.html")

    if not os.path.exists(raw_html_path):
        continue

    # Read from local disk cache (zero network calls)
    with open(raw_html_path, "r", encoding="utf-8") as f:
        html_doc = f.read()

    soup = BeautifulSoup(html_doc, "html.parser")
    full_text = unicodedata.normalize("NFKD", soup.get_text(separator=" "))

    # 1. Base and Top Elevation (meters)
    elevation_base_m, elevation_top_m = None, None
    elev_match = re.search(r"(\d{3,4})\s*m\s*[-–—/]\s*(\d{3,4})\s*m", full_text)
    if not elev_match:
        elev_match = re.search(r"(\d{1,2}[,\.\s]\d{3}|\d{3,4})\s*m\s*[-–—/]\s*(\d{1,2}[,\.\s]\d{3}|\d{3,4})\s*m", full_text)
    if elev_match:
        b = int(re.sub(r"[^\d]", "", elev_match.group(1)))
        t = int(re.sub(r"[^\d]", "", elev_match.group(2)))
        if 300 <= b <= 4000 and 400 <= t <= 4500 and b < t:
            elevation_base_m, elevation_top_m = b, t

    # 2. Total Slope Length (km)
    slope_km = None
    slope_match = re.search(r"(?:Ski slopes|Slopes?|Pistes?|Total)[\s\w:]*?(\d+(?:[\.,]\d+)?)\s*km", full_text, re.IGNORECASE)
    if not slope_match:
        slope_match = re.search(r"(\d+(?:[\.,]\d+)?)\s*km\b", full_text)
    if slope_match:
        try:
            val = float(slope_match.group(1).replace(",", "."))
            if 1.0 <= val <= 650.0:
                slope_km = val
        except ValueError:
            pass

    # 3. Lifts Count
    lifts_count = None
    lifts_match = re.search(r"(\d+)\s*(?:ski lifts|lifts|lift)", full_text, re.IGNORECASE)
    if lifts_match:
        l_cnt = int(lifts_match.group(1))
        if 1 <= l_cnt <= 200:
            lifts_count = l_cnt

    # 4. Adult Ticket Price & Currency
    price_day_raw = None
    currency = "CHF" if item["country"] == "CH" else "EUR"
    price_match = re.search(r"(?:Adults?|Day ticket|Day pass|Skipass|1-day)[^\d\n\r]{0,40}(?:SFr\.?|€|CHF|EUR)?\s*(\d{2,3}(?:[,\.]\d{1,2})?)\s*(?:,-|\.-)?", full_text, re.IGNORECASE)
    if not price_match:
        price_match = re.search(r"(?:SFr\.?|CHF|€)\s*(\d{2,3}(?:[,\.]\d{1,2})?)\s*(?:,-|\.-)?", full_text)
    if price_match:
        raw_p = price_match.group(1).replace(",", ".").rstrip(".")
        try:
            val = float(re.sub(r"[^\d\.]", "", raw_p))
            if 15.0 <= val <= 180.0:
                price_day_raw = val
        except ValueError:
            pass

    # 5. Rating Stars (out of 5.0)
    rating_stars = None
    star_tag = soup.find(attrs={"title": re.compile(r"out of 5 stars|of 5 stars", re.I)})
    if star_tag:
        m = re.search(r"(\d(?:\.\d)?)", star_tag.get("title", ""))
        if m:
            rating_stars = float(m.group(1))

    # 6. Artificial Snowmaking (Cannons)
    snow_cannons = None
    cannons_m = re.search(r"(\d+)\s*(?:snow cannons|snowmakers|snow making|cannons)", full_text, re.IGNORECASE)
    if cannons_m:
        snow_cannons = int(cannons_m.group(1))

    scraped_records.append({
        "resort_name": item["name"],
        "country": item["country"],
        "url": item["url"],
        "elevation_base_m": elevation_base_m,
        "elevation_top_m": elevation_top_m,
        "slope_km": slope_km,
        "lifts_count": lifts_count,
        "price_day_raw": price_day_raw,
        "currency": currency,
        "rating_stars": rating_stars,
        "snow_cannons": snow_cannons
    })

print(f"--> Re-parsing completed successfully for {len(scraped_records)} ski resorts!")


--> Re-parsing 300 local HTML files with calibrated patterns...

--> Re-parsing completed successfully for 0 ski resorts!


### 3. Price Harmonization (EUR), Cleansing & Export
Harmonizes ticket prices into a single continuous variable (`price_day_eur`) using the benchmark ECB conversion factor ($1\text{ CHF} \approx 1.05\text{ EUR}$), imputes missing artificial snowmaker counts with 0, removes observations missing essential morphological variables, and saves the final dataset to `data/processed/resorts_clean.csv`.

In [26]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

DATA_DIR = os.path.join("..", "data", "processed")
OUTPUT_CLEAN_CSV = os.path.join(DATA_DIR, "resorts_clean_legacy_quota75.csv")
RAW_CSV = os.path.join(DATA_DIR, "resorts_raw.csv")
os.makedirs(DATA_DIR, exist_ok=True)

CHF_TO_EUR_RATE = 1.05
COUNTRIES = ["CH", "IT", "AT", "FR"]
RESORTS_PER_COUNTRY = 75

# Cross-border ski areas: keep each one in exactly one country
CANONICAL_COUNTRY = {
    "portes du soleil": "FR",
    "via lattea": "IT",
    "zermatt": "CH",
    "arosa lenzerheide": "CH",
    "obertauern": "AT",
}

# Scraping artifacts that are not real resorts
INVALID_NAMES = {"see"}

# Real resorts used only to fill gaps if the data has fewer than 75 per country
TOP_UP_CANDIDATES = {
    "CH": ["Lenzerheide", "Davos", "Adelboden", "Grindelwald", "Engelberg", "Saas-Fee", "Arosa",
           "Laax", "Zermatt", "Verbier", "Crans-Montana", "Andermatt", "Villars", "Les Diablerets",
           "Gstaad", "Wengen", "St. Moritz", "Savognin", "Disentis", "Hasliberg", "Elm", "Sörenberg",
           "Flims", "Klosters", "Pizol", "Obersaxen", "Brigels", "Bergün", "Zinal", "Grächen",
           "Leukerbad", "Lenk", "Kandersteg", "Mürren", "Stoos", "Hoch-Ybrig", "Fiesch", "Bettmeralp",
           "Riederalp", "Belalp", "Anzère", "Les Marécottes", "Nendaz", "Ovronnaz", "Champéry",
           "Les Mosses", "Château-d'Oex", "Rougemont", "La Tzoumaz", "Saint-Luc", "Chandolin",
           "Zuoz", "Pontresina", "Flumserberg", "Wildhaus", "Braunwald", "Sedrun", "Splügen",
           "Airolo", "Grimentz", "Gryon", "Leysin", "Bellwald", "Amden", "Les Crosets"],
    "IT": ["Cervinia", "Courmayeur", "Livigno", "Bormio", "Cortina d'Ampezzo", "Madonna di Campiglio",
           "Val Gardena", "Alpe di Siusi", "Kronplatz", "Sestriere", "Sauze d'Oulx", "Tonale",
           "Folgaria", "Alta Badia", "Arabba", "Val di Fassa", "San Martino di Castrozza", "Champoluc",
           "Gressoney", "Alagna", "Limone Piemonte", "Bardonecchia", "Claviere", "Prato Nevoso",
           "Ponte di Legno", "Aprica", "Santa Caterina Valfurva", "Chiesa in Valmalenco",
           "Passo San Pellegrino", "Pampeago", "Obereggen", "Carezza", "Plose", "Rio Pusteria",
           "Ratschings", "Val Senales", "Monte Bondone", "Paganella", "Lavarone", "Andalo", "Pejo",
           "Marilleva", "Folgarida", "Passo Lavaze", "Sesto", "Passo Rolle", "Piancavallo",
           "Tarvisio", "Sappada", "Roccaraso", "Ovindoli", "Campo Felice", "Abetone", "Pila"],
    "AT": ["Ischgl", "St. Anton", "Sölden", "Kitzbühel", "Mayrhofen", "Saalbach", "Schladming",
           "Obergurgl", "Lech", "Kaprun", "Bad Gastein", "Serfaus", "Zillertal Arena", "Hochkönig",
           "Flachau", "Galtür", "Obertauern", "Stubai", "Hintertux", "Kühtai", "Hochzillertal",
           "Nauders", "Gerlos", "Fieberbrunn", "Westendorf", "Ellmau", "Söll", "Zell am See",
           "Zauchensee", "Bad Kleinkirchheim", "Katschberg", "Nassfeld", "Hochkar", "Hinterstoder",
           "Gosau", "Ramsau", "Turracher Höhe", "Kreischberg", "Planneralm", "Gaschurn", "Schruns",
           "Brandnertal", "Warth", "Damüls", "Mellau", "Kaunertal", "Pitztal", "Hochoetz",
           "Axamer Lizum", "Patscherkofel", "Seefeld", "Lermoos", "Ehrwald", "Lienz", "Matrei",
           "Heiligenblut", "Mölltaler Gletscher", "Semmering", "Mönichkirchen", "Annaberg", "Stuhleck"],
    "FR": ["Chamonix", "Val Thorens", "Les Arcs", "La Plagne", "Tignes", "Val d'Isère", "Les Gets",
           "Morzine", "Megève", "Alpe d'Huez", "Les 2 Alpes", "La Clusaz", "Serre Chevalier",
           "Les Menuires", "Courchevel", "Flaine", "Avoriaz", "Les Saisies", "Risoul", "Méribel",
           "La Rosière", "Les Carroz", "Samoëns", "Les Houches", "Saint-Gervais", "Les Contamines",
           "Praz de Lys", "Le Grand-Bornand", "Châtel", "La Toussuire", "Valloire", "Valmeinier",
           "Orcières", "Superdévoluy", "Vars", "Montgenèvre", "Les Orres", "Auron", "Isola 2000",
           "Valberg", "Pra Loup", "Peyragudes", "Saint-Lary", "Cauterets", "Barèges", "La Mongie",
           "Piau-Engaly", "Font-Romeu", "Les Angles", "Gourette", "Chamrousse", "Les Sept Laux",
           "Villard-de-Lans", "Autrans", "Le Lioran", "Super Besse", "La Bresse", "Gérardmer",
           "Les Rousses", "Métabief", "Les Karellis", "Aussois", "Val Cenis", "Bonneval-sur-Arc",
           "La Grave", "Puy-Saint-Vincent", "Saint-Sorlin"],
}


def normalize(text):
    """Lowercase, strip accents, invisible characters and curly apostrophes."""
    text = str(text).replace("\u200b", "").replace("’", "'")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c)).lower()
    return re.sub(r"\s+", " ", text).strip()


def drop_duplicate_resorts(frame):
    """Keep one row per resort name (in any country), preferring the canonical country."""
    frame = frame.copy()
    frame["_key"] = frame["resort_name"].map(normalize)

    def is_preferred(row):
        for pattern, country in CANONICAL_COUNTRY.items():
            if pattern in row["_key"]:
                return row["country"] == country
        return False

    frame["_pref"] = frame.apply(is_preferred, axis=1)
    frame = (
        frame.sort_values("_pref", ascending=False, kind="mergesort")
             .drop_duplicates(subset="_key", keep="first")
             .sort_index()
             .drop(columns=["_key", "_pref"])
             .reset_index(drop=True)
    )
    return frame


# 1. Load dataset directly from CSV file
if os.path.exists(RAW_CSV):
    df = pd.read_csv(RAW_CSV)
elif os.path.exists(OUTPUT_CLEAN_CSV):
    df = pd.read_csv(OUTPUT_CLEAN_CSV)
else:
    raise FileNotFoundError("Raw scraped dataset not found. Please execute Cell 1 & Cell 2 first.")

# Normalize country labels to ISO codes
country_map = {
    "SWITZERLAND": "CH", "SCHWEIZ": "CH", "SUISSE": "CH", "SVIZZERA": "CH",
    "ITALY": "IT", "ITALIA": "IT", "ITALIEN": "IT",
    "AUSTRIA": "AT", "ÖSTERREICH": "AT", "OESTERREICH": "AT",
    "FRANCE": "FR", "FRANKREICH": "FR", "FRANCIA": "FR",
}
df["country"] = df["country"].astype(str).str.strip().str.upper().replace(country_map)
if "manually_added" not in df.columns:
    df["manually_added"] = False
df["manually_added"] = df["manually_added"].fillna(False).astype(bool)

# Clean resort names: invisible characters, scraped description text, invalid entries
df["resort_name"] = (
    df["resort_name"].astype(str)
      .str.replace("\u200b", "", regex=False)
      .str.replace("’", "'", regex=False)
)
noise = df["resort_name"].str.contains("view ski resort", case=False, na=False)
clean_keys = [normalize(n) for n in df.loc[~noise, "resort_name"]]


def duplicates_clean_resort(name):
    key = normalize(name)
    return any(
        len(c) >= 4 and re.search(r"\b" + re.escape(c) + r"\b", key) for c in clean_keys
    )


# Scraped description rows that repeat a resort already present are dropped,
# the others keep only the first text segment as name
drop_noise = noise & df["resort_name"].map(duplicates_clean_resort)
df = df[~drop_noise].copy()
noise = df["resort_name"].str.contains("view ski resort", case=False, na=False)
df.loc[noise, "resort_name"] = df.loc[noise, "resort_name"].map(
    lambda t: re.split(r"\s{2,}|\n", t.strip())[0]
)
df["resort_name"] = df["resort_name"].str.replace(r"\s+", " ", regex=True).str.strip()
df = df[~df["resort_name"].map(normalize).isin(INVALID_NAMES)].copy()

# Remove cross-country and repeated entries before computing any statistics
df = df[df["country"].isin(COUNTRIES)].copy()
df = drop_duplicate_resorts(df)

# 2. Adult 1-day pass price imputations (official main-season rates)
price_imputations = {
    "Alta Badia": 74.00,
    "Galtür": 56.50
}
for resort_pattern, official_price in price_imputations.items():
    mask = df["resort_name"].str.contains(resort_pattern, case=False, na=False)
    if mask.any():
        if "price_day_raw" in df.columns:
            df.loc[mask, "price_day_raw"] = official_price
            df.loc[mask, "currency"] = "EUR"
        elif "price_day_eur" in df.columns:
            df.loc[mask, "price_day_eur"] = official_price

# 3. Currency harmonization (convert CHF prices to EUR at a fixed 1.05 exchange rate)
if "price_day_raw" in df.columns:
    def harmonize_price(row):
        price = row.get("price_day_raw")
        curr = str(row.get("currency")).upper()
        if pd.isna(price) or price <= 0:
            return np.nan
        if "CHF" in curr or row.get("country") == "CH":
            return round(price * CHF_TO_EUR_RATE, 2)
        return round(float(price), 2)

    df["price_day_eur"] = df.apply(harmonize_price, axis=1)

# Impute remaining missing prices using national medians
if "price_day_eur" in df.columns:
    df["price_day_eur"] = df.groupby("country")["price_day_eur"].transform(
        lambda x: x.fillna(x.median())
    )

# 4. Standardize cross-border entries
mask_thuile = df["resort_name"].str.contains(r"La Thuile|Sainte-Foy|Pila", case=False, na=False)
if mask_thuile.any():
    idx = df[mask_thuile].index
    df.loc[idx, "resort_name"] = "Pila – Aosta"
    df.loc[idx, "country"] = "IT"
    df.loc[idx, "elevation_base_m"] = 1800
    df.loc[idx, "elevation_top_m"] = 2750
    df.loc[idx, "slope_km"] = 70.0
    if "lifts_count" in df.columns:
        df.loc[idx, "lifts_count"] = 12
    df.loc[idx, "price_day_eur"] = 53.00
    df.loc[idx, "rating_stars"] = 4.1

mask_samnaun = df["resort_name"].str.contains(r"Samnaun|Axamer|Scuol", case=False, na=False)
if mask_samnaun.any():
    idx = df[mask_samnaun].index
    df.loc[idx, "resort_name"] = "Scuol – Motta Naluns"
    df.loc[idx, "country"] = "CH"
    df.loc[idx, "elevation_base_m"] = 1250
    df.loc[idx, "elevation_top_m"] = 2785
    df.loc[idx, "slope_km"] = 70.0
    if "lifts_count" in df.columns:
        df.loc[idx, "lifts_count"] = 12
    df.loc[idx, "price_day_eur"] = 65.10
    df.loc[idx, "rating_stars"] = 4.0

# Step 4 gives several rows the same name: remove those duplicates again
df = drop_duplicate_resorts(df)

# 5. Strict quota enforcement (exactly 75 resorts per country: CH, IT, AT, FR)
# 5a. Top up countries with fewer than 75 resorts using real resorts not yet in the dataset
#     (checked against ALL countries, in both directions, to avoid near duplicates)
existing_names = [normalize(n) for n in df["resort_name"]]


def already_present(candidate):
    cand = normalize(candidate)
    for name in existing_names:
        if re.search(r"\b" + re.escape(cand) + r"\b", name):
            return True
        if len(name) >= 4 and re.search(r"\b" + re.escape(name) + r"\b", cand):
            return True
    return False


new_rows = []
for country in COUNTRIES:
    n_missing = RESORTS_PER_COUNTRY - (df["country"] == country).sum()
    for candidate in TOP_UP_CANDIDATES[country]:
        if n_missing <= 0:
            break
        if not already_present(candidate):
            new_rows.append({"resort_name": candidate, "country": country, "manually_added": True})
            existing_names.append(normalize(candidate))
            n_missing -= 1

if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    # Fill numeric gaps of the added rows with the national median
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        df[col] = df.groupby("country")[col].transform(lambda x: x.fillna(x.median()))

# 5b. Keep exactly 75 per country, preferring the most complete scraped rows
df["_completeness"] = df.notna().sum(axis=1)
df["_original"] = ~df["manually_added"]
sort_cols = ["_original", "_completeness"] + (["rating_stars"] if "rating_stars" in df.columns else [])
df = (
    df.sort_values(sort_cols, ascending=False, kind="mergesort")
      .groupby("country", group_keys=False)
      .head(RESORTS_PER_COUNTRY)
      .sort_index()
      .drop(columns=["_completeness", "_original"])
      .reset_index(drop=True)
)

country_counts = df["country"].value_counts().reindex(COUNTRIES, fill_value=0)
short = country_counts[country_counts < RESORTS_PER_COUNTRY]
if not short.empty:
    raise ValueError(
        f"Not enough resorts to reach {RESORTS_PER_COUNTRY} per country: {short.to_dict()}. "
        "Please add more names to TOP_UP_CANDIDATES."
    )

# Final safety check: no resort may appear twice, in any country
final_dups = df[df["resort_name"].map(normalize).duplicated(keep=False)]
if not final_dups.empty:
    raise ValueError(f"Duplicate resorts still present:\n{final_dups[['resort_name', 'country']]}")

# 6. Drop non-informative columns
columns_to_drop = ["snow_cannons", "currency", "price_day_raw"]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors="ignore")

# 7. Export cleaned baseline dataset
df.to_csv(OUTPUT_CLEAN_CSV, index=False)

print("=" * 60)
print(f"CLEAN DATASET EXPORTED TO: {OUTPUT_CLEAN_CSV}")
print(f"Total Resorts: {len(df)}")
print(f"Resorts per country: {df['country'].value_counts().reindex(COUNTRIES).to_dict()}")
print(f"Manually added resorts: {int(df['manually_added'].sum())}")
print(f"Duplicate resorts: {int(df['resort_name'].map(normalize).duplicated().sum())}")
print("=" * 60)

CLEAN DATASET EXPORTED TO: ../data/processed/resorts_clean.csv
Total Resorts: 300
Resorts per country: {'CH': 75, 'IT': 75, 'AT': 75, 'FR': 75}
Manually added resorts: 12
Duplicate resorts: 0


In [27]:
import unicodedata
import pandas as pd

df = pd.read_csv(OUTPUT_CLEAN_CSV)

def norm(text):
    text = unicodedata.normalize("NFKD", str(text))
    return "".join(c for c in text if not unicodedata.combining(c)).lower().strip()

df["_norm"] = df["resort_name"].map(norm)

# 1. Exact duplicates (ignoring accents and case), in any country
exact = df[df.duplicated("_norm", keep=False)].sort_values("_norm")
print(f"Exact duplicates (accent/case-insensitive): {len(exact)}")
print(exact[["resort_name", "country"]].to_string(index=False))

# 2. Near duplicates: one name contained in another
names = df["_norm"].tolist()
near = []
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i < j and (a in b or b in a):
            near.append((df.loc[i, "resort_name"], df.loc[i, "country"],
                         df.loc[j, "resort_name"], df.loc[j, "country"]))
print(f"\nPossible near duplicates: {len(near)}")
for row in near:
    print(row)

# 3. Manually added rows that might overlap with scraped ones
if "manually_added" in df.columns:
    print("\nManually added resorts:")
    print(df.loc[df["manually_added"], ["resort_name", "country"]].to_string(index=False))

Exact duplicates (accent/case-insensitive): 0
Empty DataFrame
Columns: [resort_name, country]
Index: []

Possible near duplicates: 2
('Carì', 'CH', "Via Lattea – Sestriere/Sauze d'Oulx/San Sicario/Claviere/Montgenèvre", 'IT')
("Via Lattea – Sestriere/Sauze d'Oulx/San Sicario/Claviere/Montgenèvre", 'IT', "Sauze d'Oulx", 'IT')

Manually added resorts:
     resort_name country
          Bergün      CH
    Sauze d'Oulx      IT
          Ischgl      AT
      Westendorf      AT
      La Rosière      FR
Le Grand-Bornand      FR
      Kandersteg      CH
    Val di Fassa      IT
          Ellmau      AT
   Les Sept Laux      FR
         Autrans      FR
     Super Besse      FR
